In [ ]:
import os
import sys
import glob
import json
import h5py
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.extraction import load_voltage_roi_transform_h5
from vip_slap2_analysis.voltage import analysis
from vip_slap2_analysis.plotting.plot_psth import plot_voltage_mean_image_response_heatmap
from vip_slap2_analysis.utils.utils import save_figure

import matplotlib.pyplot as plt
from IPython.display import display, HTML

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

## Build session registry

In [ ]:
today_str = datetime.today().strftime('%Y-%m-%d')

BASE_PATH = Path(r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics')
SAVE_PATH = Path(r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots')

TARGET_MICE = [
852835,863774
]

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

# Voltage extraction writes files like:
#   voltage_session_traces_dff_robust_f0_trial.h5
# This variant controls the filename suffix. The plotted dataset is SIGNAL below.
TRACE_VARIANT = "dff_robust_f0_trial"
SIGNAL = "dff"      # one of: "raw_f", "f0", "dff"

# Optional direct override. Leave as None to resolve from asset.derived_dir / "voltage".
SESSION_TRACE_H5 = None

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=TARGET_MICE,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions.")
display(process_df)

## Plot mean image responses for all dendrite ROIs for a given session

In [ ]:
im_colors = [
    '#c5cae9', '#ffcdd2', '#c8e6c9', '#ffe0b2',
    '#e1bee7', '#d7ccc8',
    '#9fd3f2',   # final image: distinct light blue
    '#d9d9d9'    # omission: light gray
]

In [ ]:
asset = assets[3]
print(f'Session: {asset.session_id}')
depths = [asset.metadata['dmd1_depth'],asset.metadata['dmd2_depth']]
print(f'Recorded depths: {depths[0]} \u03BCm, {depths[1]} \u03BCm below pia')

In [ ]:
dmds = [1,2]
datapath = asset.derived_dir / 'voltage' / 'voltage_mean_dff_robust_f0_trial.npz'

In [ ]:
data = np.load(datapath,allow_pickle=True)['data'][0]
dmd1_data = data[f'DMD{dmds[0]}']
dmd2_data = data[f'DMD{dmds[1]}']

In [ ]:
image_names = list(dmd1_data['image_identity'].keys())
t = data['timebase_sec']['image']

In [ ]:
for i,dmd in enumerate([dmd1_data,dmd2_data]):
    dmd_rois = dmd['roi_ids']
    print(len(dmd_rois))
    n_col = 5
    n_row = len(dmd_rois)//n_col+1
    
    fig,axes=plt.subplots(n_row,n_col,sharex = True, sharey=True,figsize=(n_col*3,n_row*3))
    
    for ii,ax in enumerate(axes.flatten()):
        try:
            im_means = []
            for iii,image in enumerate(image_names):
                im_data = dmd['image_identity'][image]['mean'][ii]
                im_data = im_data
                ax.plot(t[:-1],im_data,color=im_colors[iii])
                im_means.append(im_data)
            mean = np.mean(im_means,axis=0)
            ax.plot(t[:-1],mean,color='k')
            ax.set_title(dmd_rois[ii])
            ax.axvspan(0.0,0.25,color='lightgray',alpha=0.4,zorder=0)
            fig.suptitle()
        except:
            pass

In [ ]:
dmd['roi_ids']

### Collect sessions

In [ ]:
mds = [1, 2]

X = []
means = []
roi_metadata = []

expected_n_features = None
expected_n_images = None
expected_trace_length = None

sessions = [0,2,6,7,8,10,11,16,17,18]

for asset in np.array(assets)[sessions]:
    session_id = str(asset.session_id)
    subject_id = asset.subject_id
    metadata = asset.metadata

    datapath = (
        asset.derived_dir
        / "voltage"
        / "voltage_mean_dff_robust_f0_trial.npz"
    )

    print(f"Session: {session_id}")
    print(f"Loading: {datapath}")

    if not datapath.exists():
        print("  File not found; skipping session.\n")
        continue

    data = np.load(datapath, allow_pickle=True)["data"][0]

    for dmd_num in dmds:
        dmd_key = f"DMD{dmd_num}"

        if dmd_key not in data:
            print(f"  {dmd_key} not found; skipping.")
            continue

        dmd = data[dmd_key]
        dmd_rois = np.asarray(dmd["roi_ids"])

        # Sorting makes image concatenation deterministic within each session.
        image_names = sorted(dmd["image_identity"].keys(), key=str)

        depth = metadata.get(f"dmd{dmd_num}_depth", np.nan)

        print(
            f"  {dmd_key}: {len(dmd_rois)} ROIs, "
            f"depth = {depth} µm, "
            f"session type = {metadata.get('session_type', np.nan)}"
        )

        for roi_index, roi_id in enumerate(dmd_rois):
            try:
                image_responses = []

                for image_name in image_names:
                    mean_array = np.asarray(
                        dmd["image_identity"][image_name]["mean"]
                    )

                    im_data = np.asarray(
                        mean_array[roi_index],
                        dtype=float,
                    ).squeeze()

                    if im_data.ndim != 1:
                        raise ValueError(
                            f"Expected 1D mean trace, got {im_data.shape}"
                        )

                    # Retains your current baseline convention.
                    im_data = im_data - im_data[0]
                    image_responses.append(im_data)

                # Check that image traces have equal duration.
                image_lengths = [len(x) for x in image_responses]

                if len(set(image_lengths)) != 1:
                    raise ValueError(
                        f"Unequal image trace lengths: {image_lengths}"
                    )

                concatenated_response = np.concatenate(image_responses)
                mean_response = np.mean(
                    np.stack(image_responses, axis=0),
                    axis=0,
                )

                # PCA requires the same number of features for every ROI.
                if expected_n_features is None:
                    expected_n_features = len(concatenated_response)
                    expected_n_images = len(image_names)
                    expected_trace_length = image_lengths[0]

                if len(concatenated_response) != expected_n_features:
                    raise ValueError(
                        f"Feature length {len(concatenated_response)} "
                        f"does not match expected {expected_n_features}"
                    )

                roi_id_string = (
                    roi_id.decode()
                    if isinstance(roi_id, bytes)
                    else str(roi_id)
                )

                roi_uid = f"{session_id}__{dmd_key}__{roi_id_string}"

                X.append(concatenated_response)
                means.append(mean_response)

                roi_metadata.append(
                    {
                        # Unique observation identifiers
                        "roi_uid": roi_uid,
                        "roi_id": roi_id_string,
                        "roi_index": roi_index,
                        "dmd": dmd_key,
                        "dmd_num": dmd_num,

                        # Session identifiers
                        "session_id": session_id,
                        "subject_id": subject_id,
                        "session_number": metadata.get(
                            "session_#", np.nan
                        ),
                        "session_date": metadata.get(
                            "session_date", pd.NaT
                        ),

                        # Experimental factors
                        "depth_um": depth,
                        "session_type": metadata.get(
                            "session_type", np.nan
                        ),
                        "stimulus": metadata.get(
                            "stimulus", np.nan
                        ),
                        "paradigm": metadata.get(
                            "paradigm", np.nan
                        ),
                        "indicator1": metadata.get(
                            "indicator1", np.nan
                        ),
                        "quality": metadata.get(
                            "quality", np.nan
                        ),
                        "flags": metadata.get(
                            "flags", np.nan
                        ),

                        # Useful response-structure information
                        "n_images": len(image_names),
                        "trace_length": image_lengths[0],
                        "image_names": tuple(map(str, image_names)),
                        "source_path": str(datapath),
                    }
                )

            except Exception as exc:
                print(
                    f"    Could not extract ROI {roi_id!r} "
                    f"from {dmd_key}: {exc}"
                )

    print()

## Compute PCA

In [ ]:
X = np.asarray(X, dtype=float)
means = np.asarray(means, dtype=float)
roi_meta = pd.DataFrame(roi_metadata)

print("X shape:", X.shape)
print("Mean-response shape:", means.shape)
print("Metadata shape:", roi_meta.shape)

assert X.shape[0] == means.shape[0]
assert X.shape[0] == len(roi_meta)
assert roi_meta["roi_uid"].is_unique
assert np.isfinite(X).all()

display(roi_meta.head())

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler(with_mean=True, with_std=True)
Xz = scaler.fit_transform(X)

n_components = min(10, Xz.shape[0] - 1, Xz.shape[1])

pca = PCA(n_components=n_components)
Xpca = pca.fit_transform(Xz)

for pc_index in range(Xpca.shape[1]):
    roi_meta[f"PC{pc_index + 1}"] = Xpca[:, pc_index]

print(
    "Variance explained by PC1–PC5:",
    pca.explained_variance_ratio_[:5],
)

## Plot PCA

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

sns.scatterplot(
    data=roi_meta,
    x="PC1",
    y="PC2",
    hue="session_type",
    style="dmd",
    s=38,
    alpha=0.75,
    ax=ax,
)

ax.set_xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]:.1%})"
)
ax.set_ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]:.1%})"
)

ax.legend(title=None, 
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

sns.despine()
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

sns.scatterplot(
    data=roi_meta,
    x="PC1",
    y="PC2",
    hue="session_id",
    style="dmd",
    palette='tab20',
    s=30,
    alpha=0.75,
    legend=True,
    ax=ax,
)
ax.set_title("PCA coordinates by recording session")
sns.despine()
ax.legend(ncols=3,fontsize=8,frameon=False)
fig.tight_layout()

In [ ]:
session_centroids = (
    roi_meta
    .groupby(
        ["session_type"],
        as_index=False,
    )
    .agg(
        PC1=("PC1", "mean"),
        PC2=("PC2", "mean"),
        n_rois=("roi_uid", "size"),
    )
)

fig, ax = plt.subplots(figsize=(5.5, 4.5))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

sns.scatterplot(
    data=roi_meta,
    x="PC1",
    y="PC2",
    hue="session_type",
    style="dmd",
    s=32,
    alpha=0.5,
    ax=ax,legend=False
)

sns.scatterplot(
    data=session_centroids,
    x="PC1",
    y="PC2",
    hue="session_type",
    size="n_rois",
    sizes=(100, 250),
    edgecolor="black",
    linewidth=1,
    ax=ax,legend=False
)
# ax.legend(frameon=False,fontsize=12)
ax.set_title("ROI scores and session-type centroids")
sns.despine()
fig.tight_layout()

In [ ]:
pc_summary = (
    roi_meta
    .groupby(["session_type", "dmd"], dropna=False)
    .agg(
        n_rois=("roi_uid", "size"),
        n_sessions=("session_id", "nunique"),
        n_mice=("subject_id", "nunique"),
        mean_depth_um=("depth_um", "mean"),
        PC1_mean=("PC1", "mean"),
        PC1_sd=("PC1", "std"),
        PC2_mean=("PC2", "mean"),
        PC2_sd=("PC2", "std"),
    )
    .reset_index()
)

display(pc_summary)

In [ ]:
from scipy.stats import spearmanr


valid = roi_meta[["depth_um", "PC1", "PC2"]].dropna()

for pc_name in ["PC1", "PC2"]:
    rho, p = spearmanr(valid["depth_um"], valid[pc_name])
    print(f"{pc_name}: ROI-level rho={rho:.3f}, p={p:.3g}")

In [ ]:
import statsmodels.formula.api as smf


model_df = roi_meta[
    [
        "PC1",
        "depth_um",
        "session_type",
        "session_id",
        "subject_id",
    ]
].dropna().copy()

model_df["session_type"] = model_df["session_type"].astype("category")

model = smf.mixedlm(
    "PC1 ~ depth_um + C(session_type)",
    data=model_df,
    groups=model_df["session_id"],
)

result = model.fit()
print(result.summary())

## Explained variance

In [ ]:
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
pcs = np.arange(1, len(explained_var) + 1)

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

ax.bar(pcs, explained_var, color='0.7', edgecolor='k',lw=2)
ax.plot(pcs, cumulative_var, 'o-', color='k', lw=1.5, ms=4)
ax.set_xlabel('Principal component')
ax.set_ylabel('Variance explained')
ax.set_xticks(pcs)
ax.set_ylim(0, 1.02)
ax.set_title('PCA explained variance')
sns.despine()

for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)

fig.tight_layout()

filen = 'PCA_explained_variance'
# save_figure(fig,os.path.join(savepath,filen),formats=['.pdf','.png'],dpi=300)

### Image response motifs

In [ ]:
dmds = [1, 2]

# Each row of X will be one mean image response from one ROI.
X = []
response_metadata = []

expected_trace_length = None

sessions = [0, 2, 6, 7, 8, 10, 11, 16, 17, 18]

for asset in np.array(assets)[sessions]:
    session_id = str(asset.session_id)
    subject_id = asset.subject_id
    metadata = asset.metadata

    datapath = (
        asset.derived_dir
        / "voltage"
        / "voltage_mean_dff_robust_f0_trial.npz"
    )

    print(f"Session: {session_id}")
    print(f"Loading: {datapath}")

    if not datapath.exists():
        print("  File not found; skipping session.\n")
        continue

    data = np.load(
        datapath,
        allow_pickle=True,
    )["data"][0]

    for dmd_num in dmds:
        dmd_key = f"DMD{dmd_num}"

        if dmd_key not in data:
            print(f"  {dmd_key} not found; skipping.")
            continue

        dmd = data[dmd_key]
        dmd_rois = np.asarray(dmd["roi_ids"])

        # Deterministic ordering, although responses are no longer concatenated.
        image_names = sorted(
            dmd["image_identity"].keys(),
            key=str,
        )

        depth = metadata.get(
            f"dmd{dmd_num}_depth",
            np.nan,
        )

        print(
            f"  {dmd_key}: {len(dmd_rois)} ROIs, "
            f"{len(image_names)} images, "
            f"depth = {depth} µm, "
            f"session type = {metadata.get('session_type', np.nan)}"
        )

        for roi_index, roi_id in enumerate(dmd_rois):
            roi_id_string = (
                roi_id.decode()
                if isinstance(roi_id, bytes)
                else str(roi_id)
            )

            roi_uid = (
                f"{session_id}__"
                f"{dmd_key}__"
                f"{roi_id_string}"
            )

            for image_index, image_name in enumerate(image_names):
                try:
                    mean_array = np.asarray(
                        dmd["image_identity"][image_name]["mean"]
                    )

                    im_data = np.asarray(
                        mean_array[roi_index],
                        dtype=float,
                    ).squeeze()

                    if im_data.ndim != 1:
                        raise ValueError(
                            "Expected a one-dimensional mean trace, "
                            f"got shape {im_data.shape}"
                        )

                    if not np.all(np.isfinite(im_data)):
                        raise ValueError(
                            "Trace contains non-finite values"
                        )

                    # Retains the current baseline convention.
                    im_data = im_data - im_data[0]

                    # All PCA observations must have the same number
                    # of temporal features.
                    if expected_trace_length is None:
                        expected_trace_length = len(im_data)

                    if len(im_data) != expected_trace_length:
                        raise ValueError(
                            f"Trace length {len(im_data)} does not "
                            f"match expected length "
                            f"{expected_trace_length}"
                        )

                    image_name_string = (
                        image_name.decode()
                        if isinstance(image_name, bytes)
                        else str(image_name)
                    )

                    response_uid = (
                        f"{roi_uid}__"
                        f"image-{image_name_string}"
                    )

                    # One PCA observation per ROI × image response.
                    X.append(im_data)

                    response_metadata.append(
                        {
                            # Unique response identifier
                            "response_uid": response_uid,

                            # ROI identifiers
                            "roi_uid": roi_uid,
                            "roi_id": roi_id_string,
                            "roi_index": roi_index,
                            "dmd": dmd_key,
                            "dmd_num": dmd_num,

                            # Image identifiers
                            "image_name": image_name_string,
                            "image_index": image_index,
                            "n_images": len(image_names),
                            "image_names": tuple(
                                map(str, image_names)
                            ),

                            # Session identifiers
                            "session_id": session_id,
                            "subject_id": subject_id,
                            "session_number": metadata.get(
                                "session_#",
                                np.nan,
                            ),
                            "session_date": metadata.get(
                                "session_date",
                                pd.NaT,
                            ),

                            # Experimental factors
                            "depth_um": depth,
                            "session_type": metadata.get(
                                "session_type",
                                np.nan,
                            ),
                            "stimulus": metadata.get(
                                "stimulus",
                                np.nan,
                            ),
                            "paradigm": metadata.get(
                                "paradigm",
                                np.nan,
                            ),
                            "indicator1": metadata.get(
                                "indicator1",
                                np.nan,
                            ),
                            "quality": metadata.get(
                                "quality",
                                np.nan,
                            ),
                            "flags": metadata.get(
                                "flags",
                                np.nan,
                            ),

                            # Response information
                            "trace_length": len(im_data),
                            "source_path": str(datapath),
                        }
                    )

                except Exception as exc:
                    print(
                        f"    Could not extract ROI {roi_id!r}, "
                        f"image {image_name!r} from {dmd_key}: "
                        f"{exc}"
                    )

    print()


# Shape:
#   n_roi_image_responses × n_time_samples
if len(X) == 0:
    raise RuntimeError("No valid ROI-image responses were extracted.")

X = np.stack(X, axis=0)
response_metadata = pd.DataFrame(response_metadata)

if len(response_metadata) != X.shape[0]:
    raise RuntimeError(
        "Metadata and response matrix have different row counts."
    )

print(f"X shape: {X.shape}")
print(f"Metadata shape: {response_metadata.shape}")
print(
    f"Unique ROIs: "
    f"{response_metadata['roi_uid'].nunique()}"
)
print(
    f"Unique sessions: "
    f"{response_metadata['session_id'].nunique()}"
)
print(
    f"Unique subjects: "
    f"{response_metadata['subject_id'].nunique()}"
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import resample_poly
from sklearn.decomposition import PCA


# ---------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------

# Approximate sampling rate of the mean response traces.
sample_rate_hz = 10_800

# Set this to the actual start of the peri-image response window.
window_start_sec = -0.25

# Downsampling is strongly recommended for mean PSTHs.
# 10 gives an effective sampling rate of approximately 1080 Hz.
downsample_factor = 10

# Number of PCs to calculate.
n_components = 20

# Options:
#   "amplitude" : preserves both response magnitude and waveform shape
#   "shape"     : normalizes each response before PCA
analysis_mode = "amplitude"

# ASAP7 is quenched by depolarization. Flip the sign so that
# depolarizing responses appear upward in plots.
invert_asap7 = True


# ---------------------------------------------------------------------
# Validate inputs
# ---------------------------------------------------------------------

X_raw = np.asarray(X, dtype=float)
pca_metadata = response_metadata.copy().reset_index(drop=True)

if X_raw.ndim != 2:
    raise ValueError(
        f"X must be two-dimensional, but has shape {X_raw.shape}"
    )

if len(pca_metadata) != X_raw.shape[0]:
    raise ValueError(
        "The number of metadata rows does not match the number "
        f"of observations in X: {len(pca_metadata)} versus {X_raw.shape[0]}"
    )

print(f"Initial X shape: {X_raw.shape}")


# ---------------------------------------------------------------------
# Remove invalid observations
# ---------------------------------------------------------------------

finite_rows = np.all(np.isfinite(X_raw), axis=1)

if not np.all(finite_rows):
    print(
        f"Removing {np.sum(~finite_rows)} responses containing "
        "NaN or infinite values."
    )

X_valid = X_raw[finite_rows]
pca_metadata = (
    pca_metadata.loc[finite_rows]
    .reset_index(drop=True)
)

print(f"Valid X shape: {X_valid.shape}")


# ---------------------------------------------------------------------
# Convert ASAP7 responses to depolarization-up convention
# ---------------------------------------------------------------------

if invert_asap7:
    X_valid = -X_valid


# ---------------------------------------------------------------------
# Construct the original time axis
# ---------------------------------------------------------------------

time_sec_original = (
    window_start_sec
    + np.arange(X_valid.shape[1]) / sample_rate_hz
)


# ---------------------------------------------------------------------
# Downsample the mean responses
# ---------------------------------------------------------------------

if downsample_factor > 1:
    X_downsampled = resample_poly(
        X_valid,
        up=1,
        down=downsample_factor,
        axis=1,
    )

    effective_sample_rate_hz = (
        sample_rate_hz / downsample_factor
    )

    time_sec = (
        window_start_sec
        + np.arange(X_downsampled.shape[1])
        / effective_sample_rate_hz
    )

else:
    X_downsampled = X_valid.copy()
    effective_sample_rate_hz = sample_rate_hz
    time_sec = time_sec_original.copy()

print(f"Downsampled X shape: {X_downsampled.shape}")
print(
    f"Effective sampling rate: "
    f"{effective_sample_rate_hz:.1f} Hz"
)


# ---------------------------------------------------------------------
# Select amplitude-preserving or shape-focused representation
# ---------------------------------------------------------------------

if analysis_mode == "amplitude":
    X_pca = X_downsampled.copy()
    retained_rows = np.ones(
        X_downsampled.shape[0],
        dtype=bool,
    )

elif analysis_mode == "shape":
    response_norm = np.linalg.norm(
        X_downsampled,
        axis=1,
    )

    # Exclude very weak responses before normalization because dividing
    # by a small norm can turn noise into an apparent waveform motif.
    positive_norms = response_norm[
        np.isfinite(response_norm)
        & (response_norm > 0)
    ]

    if len(positive_norms) == 0:
        raise RuntimeError(
            "No responses have a positive finite norm."
        )

    # This is an empirical safeguard rather than a definitive
    # response-significance threshold.
    minimum_norm = np.percentile(
        positive_norms,
        10,
    )

    retained_rows = (
        np.isfinite(response_norm)
        & (response_norm > minimum_norm)
    )

    X_pca = (
        X_downsampled[retained_rows]
        / response_norm[retained_rows, None]
    )

    pca_metadata = (
        pca_metadata.loc[retained_rows]
        .reset_index(drop=True)
    )

    X_downsampled = X_downsampled[retained_rows]

    print(
        f"Shape PCA retained {X_pca.shape[0]} responses "
        f"with norm > {minimum_norm:.4g}"
    )

else:
    raise ValueError(
        "analysis_mode must be either 'amplitude' or 'shape'"
    )


# ---------------------------------------------------------------------
# Fit PCA
# ---------------------------------------------------------------------

maximum_components = min(
    X_pca.shape[0],
    X_pca.shape[1],
)

n_components_used = min(
    n_components,
    maximum_components,
)

pca = PCA(
    n_components=n_components_used,
    svd_solver="auto",
    random_state=0,
)

pca_scores = pca.fit_transform(X_pca)

print()
print(f"PCA input shape: {X_pca.shape}")
print(f"PCA scores shape: {pca_scores.shape}")
print(f"PCA components shape: {pca.components_.shape}")
print(
    "Variance explained by first 5 PCs: "
    f"{pca.explained_variance_ratio_[:5]}"
)
print(
    "Cumulative variance explained by first 5 PCs: "
    f"{pca.explained_variance_ratio_[:5].sum():.3f}"
)


# ---------------------------------------------------------------------
# Add PCA scores to the metadata table
# ---------------------------------------------------------------------

for pc_index in range(pca_scores.shape[1]):
    pca_metadata[f"PC{pc_index + 1}"] = (
        pca_scores[:, pc_index]
    )

pca_metadata.head()

In [ ]:
variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratio)
pc_numbers = np.arange(1, len(variance_ratio) + 1)

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    pc_numbers,
    variance_ratio,
    marker="o",
    label="Individual PC",
)

ax.plot(
    pc_numbers,
    cumulative_variance,
    marker="o",
    label="Cumulative",
)

ax.set(
    xlabel="Principal component",
    ylabel="Fraction of variance explained",
    title=f"PCA explained variance: {analysis_mode} mode",
)

ax.set_xticks(pc_numbers)
ax.axhline(0.80, linestyle="--", linewidth=1)
ax.axhline(0.90, linestyle="--", linewidth=1)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()

In [ ]:
for threshold in [0.50, 0.75, 0.80, 0.90, 0.95]:
    n_required = (
        np.searchsorted(
            cumulative_variance,
            threshold,
        )
        + 1
    )

    if n_required <= len(cumulative_variance):
        print(
            f"{n_required:2d} PCs explain at least "
            f"{threshold:.0%} of the variance."
        )

In [ ]:
n_components_to_plot = min(
    6,
    pca.n_components_,
)

fig, axes = plt.subplots(
    n_components_to_plot,
    1,
    figsize=(8, 2.2 * n_components_to_plot),
    sharex=True,
)

axes = np.atleast_1d(axes)

for pc_index, ax in enumerate(axes):
    component = pca.components_[pc_index]

    ax.plot(
        time_sec,
        component,
        linewidth=1.5,
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1,
    )

    ax.axhline(
        0,
        linewidth=0.8,
    )

    ax.set_ylabel(f"PC{pc_index + 1}")

    ax.text(
        0.99,
        0.92,
        (
            f"{100 * pca.explained_variance_ratio_[pc_index]:.1f}% "
            "variance"
        ),
        transform=ax.transAxes,
        ha="right",
        va="top",
    )

axes[-1].set_xlabel("Time from image onset (s)")
axes[0].set_title("Temporal principal components")

fig.tight_layout()
plt.show()

In [ ]:
n_components_to_plot = min(
    6,
    pca.n_components_,
)

fig, axes = plt.subplots(
    n_components_to_plot,
    1,
    figsize=(8, 2.5 * n_components_to_plot),
    sharex=True,
)

axes = np.atleast_1d(axes)

for pc_index, ax in enumerate(axes):
    component_scale = np.sqrt(
        pca.explained_variance_[pc_index]
    )

    positive_mode = (
        pca.mean_
        + component_scale
        * pca.components_[pc_index]
    )

    negative_mode = (
        pca.mean_
        - component_scale
        * pca.components_[pc_index]
    )

    ax.plot(
        time_sec,
        pca.mean_,
        linewidth=1.5,
        label="PCA mean",
    )

    ax.plot(
        time_sec,
        positive_mode,
        linewidth=1.2,
        label="+1 SD along PC",
    )

    ax.plot(
        time_sec,
        negative_mode,
        linewidth=1.2,
        label="−1 SD along PC",
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1,
    )

    ax.set_ylabel(f"PC{pc_index + 1}")

axes[0].legend(
    frameon=False,
    ncol=3,
)

axes[-1].set_xlabel("Time from image onset (s)")
axes[0].set_title("PCA response modes")

fig.tight_layout()
plt.show()

In [ ]:
def plot_pca_scores(
    scores,
    metadata,
    *,
    pc_x=1,
    pc_y=2,
    color_by="session_type",
    alpha=0.65,
    point_size=20,
):
    """
    Plot two PCA score dimensions, colored by a metadata column.

    Parameters
    ----------
    scores : array, shape (n_observations, n_components)
        PCA score matrix.

    metadata : pandas.DataFrame
        Metadata corresponding row-for-row with scores.

    pc_x, pc_y : int
        One-indexed PC numbers.

    color_by : str
        Metadata column used to color observations.
    """
    x_index = pc_x - 1
    y_index = pc_y - 1

    if color_by not in metadata.columns:
        raise KeyError(
            f"{color_by!r} is not present in metadata."
        )

    values = metadata[color_by]

    fig, ax = plt.subplots(figsize=(7, 6))

    is_numeric = pd.api.types.is_numeric_dtype(values)

    if is_numeric:
        numeric_values = pd.to_numeric(
            values,
            errors="coerce",
        )

        valid = numeric_values.notna().to_numpy()

        scatter = ax.scatter(
            scores[valid, x_index],
            scores[valid, y_index],
            c=numeric_values.loc[valid],
            s=point_size,
            alpha=alpha,
        )

        colorbar = fig.colorbar(
            scatter,
            ax=ax,
        )

        colorbar.set_label(color_by)

    else:
        category_values = (
            values.fillna("missing")
            .astype(str)
        )

        categories = sorted(
            category_values.unique()
        )

        for category in categories:
            mask = (
                category_values == category
            ).to_numpy()

            ax.scatter(
                scores[mask, x_index],
                scores[mask, y_index],
                s=point_size,
                alpha=alpha,
                label=category,
            )

        ax.legend(
            title=color_by,
            frameon=False,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
        )

    ax.axhline(
        0,
        linewidth=0.8,
    )

    ax.axvline(
        0,
        linewidth=0.8,
    )

    ax.set(
        xlabel=(
            f"PC{pc_x} "
            f"({100 * pca.explained_variance_ratio_[x_index]:.1f}%)"
        ),
        ylabel=(
            f"PC{pc_y} "
            f"({100 * pca.explained_variance_ratio_[y_index]:.1f}%)"
        ),
        title=f"PCA scores colored by {color_by}",
    )

    fig.tight_layout()
    plt.show()

In [ ]:
plot_pca_scores(
    pca_scores,
    pca_metadata,
    color_by="session_type",
)

In [ ]:
plot_pca_scores(
    pca_scores,
    pca_metadata,
    color_by="depth_um",
)

In [ ]:
plot_pca_scores(
    pca_scores,
    pca_metadata,
    color_by="image_name",
)

In [ ]:
plot_pca_scores(
    pca_scores,
    pca_metadata,
    color_by="session_id",
)

In [ ]:
def plot_pc_extreme_responses(
    X_responses,
    scores,
    time_sec,
    pca_model,
    *,
    pc=1,
    quantile=0.10,
    max_individual_traces=50,
):
    """
    Plot real responses at the positive and negative extremes
    of one principal component.
    """
    pc_index = pc - 1
    pc_scores = scores[:, pc_index]

    lower_threshold = np.quantile(
        pc_scores,
        quantile,
    )

    upper_threshold = np.quantile(
        pc_scores,
        1 - quantile,
    )

    negative_mask = pc_scores <= lower_threshold
    positive_mask = pc_scores >= upper_threshold

    negative_indices = np.flatnonzero(
        negative_mask
    )

    positive_indices = np.flatnonzero(
        positive_mask
    )

    rng = np.random.default_rng(0)

    if len(negative_indices) > max_individual_traces:
        negative_plot_indices = rng.choice(
            negative_indices,
            size=max_individual_traces,
            replace=False,
        )
    else:
        negative_plot_indices = negative_indices

    if len(positive_indices) > max_individual_traces:
        positive_plot_indices = rng.choice(
            positive_indices,
            size=max_individual_traces,
            replace=False,
        )
    else:
        positive_plot_indices = positive_indices

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 4),
        sharex=True,
        sharey=True,
    )

    for response in X_responses[negative_plot_indices]:
        axes[0].plot(
            time_sec,
            response,
            linewidth=0.5,
            alpha=0.15,
        )

    axes[0].plot(
        time_sec,
        X_responses[negative_mask].mean(axis=0),
        linewidth=2.5,
        label="Mean",
    )

    for response in X_responses[positive_plot_indices]:
        axes[1].plot(
            time_sec,
            response,
            linewidth=0.5,
            alpha=0.15,
        )

    axes[1].plot(
        time_sec,
        X_responses[positive_mask].mean(axis=0),
        linewidth=2.5,
        label="Mean",
    )

    for ax in axes:
        ax.axvline(
            0,
            linestyle="--",
            linewidth=1,
        )

        ax.axhline(
            0,
            linewidth=0.8,
        )

        ax.set_xlabel("Time from image onset (s)")
        ax.legend(frameon=False)

    axes[0].set_ylabel(
        "Depolarization-oriented dFF"
        if invert_asap7
        else "dFF"
    )

    axes[0].set_title(
        f"Lowest {quantile:.0%} of PC{pc} scores\n"
        f"n = {negative_mask.sum()}"
    )

    axes[1].set_title(
        f"Highest {quantile:.0%} of PC{pc} scores\n"
        f"n = {positive_mask.sum()}"
    )

    variance_percent = (
        100
        * pca_model.explained_variance_ratio_[pc_index]
    )

    fig.suptitle(
        f"Observed response extremes along PC{pc} "
        f"({variance_percent:.1f}% variance)"
    )

    fig.tight_layout()
    plt.show()

In [ ]:
for pc in range(1, 5):
    plot_pc_extreme_responses(
        X_downsampled,
        pca_scores,
        time_sec,
        pca,
        pc=pc,
        quantile=0.10,
    )

In [ ]:
def plot_pc_score_bins(
    X_responses,
    scores,
    time_sec,
    *,
    pc=1,
    n_bins=5,
):
    pc_index = pc - 1
    pc_scores = scores[:, pc_index]

    bin_edges = np.quantile(
        pc_scores,
        np.linspace(0, 1, n_bins + 1),
    )

    # Prevent duplicated boundaries from causing invalid bins.
    bin_edges = np.unique(bin_edges)

    bin_ids = np.digitize(
        pc_scores,
        bin_edges[1:-1],
        right=False,
    )

    fig, ax = plt.subplots(figsize=(8, 5))

    for bin_id in range(len(bin_edges) - 1):
        mask = bin_ids == bin_id

        if not np.any(mask):
            continue

        mean_response = np.mean(
            X_responses[mask],
            axis=0,
        )

        score_mean = np.mean(
            pc_scores[mask]
        )

        ax.plot(
            time_sec,
            mean_response,
            linewidth=1.5,
            label=(
                f"Bin {bin_id + 1}: "
                f"mean score = {score_mean:.2f}, "
                f"n = {mask.sum()}"
            ),
        )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1,
    )

    ax.axhline(
        0,
        linewidth=0.8,
    )

    ax.set(
        xlabel="Time from image onset (s)",
        ylabel=(
            "Depolarization-oriented dFF"
            if invert_asap7
            else "dFF"
        ),
        title=f"Average responses across PC{pc} score bins",
    )

    ax.legend(
        frameon=False,
        fontsize=8,
    )

    fig.tight_layout()
    plt.show()

In [ ]:
plot_pc_score_bins(
    X_downsampled,
    pca_scores,
    time_sec,
    pc=1,
    n_bins=5,
)

In [ ]:
n_reconstruction_components = min(
    5,
    pca.n_components_,
)

truncated_scores = pca_scores.copy()
truncated_scores[:, n_reconstruction_components:] = 0

X_reconstructed = pca.inverse_transform(
    truncated_scores
)

rng = np.random.default_rng(0)
example_indices = rng.choice(
    X_pca.shape[0],
    size=min(6, X_pca.shape[0]),
    replace=False,
)

fig, axes = plt.subplots(
    len(example_indices),
    1,
    figsize=(8, 2.2 * len(example_indices)),
    sharex=True,
)

axes = np.atleast_1d(axes)

for ax, response_index in zip(
    axes,
    example_indices,
):
    ax.plot(
        time_sec,
        X_pca[response_index],
        linewidth=1.2,
        label="Observed",
    )

    ax.plot(
        time_sec,
        X_reconstructed[response_index],
        linewidth=1.2,
        label=(
            f"{n_reconstruction_components}-PC "
            "reconstruction"
        ),
    )

    ax.axvline(
        0,
        linestyle="--",
        linewidth=1,
    )

    metadata_row = pca_metadata.iloc[
        response_index
    ]

    ax.set_title(
        f"{metadata_row['session_id']} | "
        f"{metadata_row['dmd']} | "
        f"ROI {metadata_row['roi_id']} | "
        f"{metadata_row['image_name']}"
    )

axes[0].legend(
    frameon=False,
)

axes[-1].set_xlabel("Time from image onset (s)")

fig.tight_layout()
plt.show()